**An encoder-decoder network for neural machine translation...**
- first we need to download a dataset of English/Spanish pairs from the Tatoeba challenge dataset
- Training set is quite large so we'll use the validation set as oir training set, setting aside 20% for validation
- We will also download the test set...

In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset

In [2]:
import tokenizers

using the original valid set instead of the train set as the original train set is too huge.....

In [3]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split =['validation','test']
)

split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split['train'], split['test']

Each sample in the dataset is a dictionary containing an english text along with its spanish translation...

In [4]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

We need to tokenize this text. we use a common tokenizer as these 2 languages have many words in common. Lets train a BPE tokenizer on all the training text, both english and spanish..

tokenization: converting this text to the base units the model will use to think ......

In [49]:
def train_eng_spa(): # a generator function to iterate over all training text
    for pair in nmt_train_set:
        yield pair['source_text']
        yield pair['target_text']

In [50]:
max_length = 256 # our custom tokenizer will enable truncation and we need a max length param to truncate at......
vocab_size = 10000 # total size of our vocabulary
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>") # instantiate the model we'll use for tokenization: using a BPE model in this case
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model) # instantiate the tokenizer - giving it the model
# specify the tokenizer's normalizer - 
nmt_tokenizer.normalizer = tokenizers.normalizers.Sequence([tokenizers.normalizers.Lowercase()])
# enable padding and truncation for the normalizer
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
# the pre_tokenizer - choosing to go with byte level pretokenization in this instance so as to eliminate the possibility of unknown tokens
# the pre_tokenizer splits the sentences into words (bytes) - the maximum units that subwords can build up to
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
# specifying the trainer - bpe trainer and the special tokens that will be used
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>","<unk>","<s>","</s>"] # note that we add a sos and eos token
)
# now training the tokenizer
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

Test the trained tokenizer..

In [51]:
enc = nmt_tokenizer.encode("Blaise is my name")

In [52]:
enc

Encoding(num_tokens=5, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [53]:
enc.tokens

['bla', 'ise', 'is', 'my', 'name']

In [54]:
nmt_tokenizer.encode("I like soccer").ids

[46, 356, 3811]

In [55]:
nmt_tokenizer.encode("<s> Me gusta el fútbol").ids

[2, 178, 544, 169, 3125]

Now, we create a small utility class that will hold english texts (i.e. the source token id sequences), along with the corresponding spanish targets (i.e. the target token ID sequences), plus the corresponding attention masks. for this we create a namedtuple base class and extend it to add a to() method which will make it easy to move all tensors to the GPU

This is the utility class that we'll use to train the model and will hold all the train data

In [56]:
from collections import namedtuple

In [57]:
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]

class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device), 
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

Next, create the dataloaders:

In [58]:
nmt_train_set[0]['source_text']

'Tom tried to break up the fight.'

In [59]:
# collate function that will be returned by our dataloader
def nmt_collate_fn(batch):
    # remember according to how the dataloader works - at this instance, batch is a tuple composed of individual items returned by indexing the dataset from 0 to batchsize
    src_texts = [pair['source_text'] for pair in batch] # extracting all the source english texts
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch] # extracting the target spanish texts and adding a sos and eos tokens
    src_encodings = nmt_tokenizer.encode_batch(src_texts) # now tokenizing the source english texts
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts) # now tokenizing the spanish texts
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings]) # converting the token id lists to pytorch tensors
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings]) # converting the mask id lists to tensors
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    # now creating the named tuple that holds the training english and spanish texts
    # for the teacher training spanish texts, we get everything except the last token as the model doesn't need to see that one since training will be done at that point
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:,:-1], tgt_mask[:,:-1]) 
    # the targets are from excluding the sos token onwards
    labels = tgt_token_ids[:,1:]
    return inputs, labels

In [60]:
from torch.utils.data import DataLoader

In [61]:
batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size, collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size, collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size, collate_fn=nmt_collate_fn)

Test this out to visualise the structure and shape of the returned data:

In [62]:
t_train,t_labels = next(iter(nmt_train_loader))

In [63]:
# expect the input train data to be batch x n-dimensional and the labels to be the same batch x n-dimensional shape. the dimension depends on the sequence tokenized
t_train.src_token_ids.shape, t_train.src_mask.shape, t_train.tgt_token_ids.shape, t_train.tgt_mask.shape

(torch.Size([32, 19]),
 torch.Size([32, 19]),
 torch.Size([32, 21]),
 torch.Size([32, 21]))

In [64]:
# expect the labels to be of the same shape as the tgt_encodings since they are everything in it excluding the sos token but including the eos token
t_labels.shape

torch.Size([32, 21])

Now that we've got the data structure out of the way - lets look at the model...
- Below is the target architecture to build
- <img src="/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/resource_images/target_model.png" height=500 width=500/>

In [65]:
class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512, n_layers=2):
        super().__init__()
        # we use the same tokenizer so we shall use the same embedding layer
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        # the encoder a GRU - output of shape batch_size x seq_length x hidden_dim
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        # the decoder - a GRU - output of shape batch_size x seq_length x hidden_dim
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        # taking the decoder output and putting it to a linear layer
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, pair):
        # compute the encoder embeddings
        src_embeddings = self.embed(pair.src_token_ids)
        # compute the decoder embeddings
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        # we are using packed sequences for the encoder so we need the lengths
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = nn.utils.rnn.pack_padded_sequence(
            src_embeddings, lengths = src_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        # compute the encoder output and then extract the last hidden state which then gets fed to the decoder as the first hidden state
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        # get the outputs at each timestep for the decoder - feed them to the linear layer -> b x s x hidden -> permute to get b x hidden (vocab) x seq length
        return self.output(outputs).permute(0, 2, 1)

In [66]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [67]:
torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to(device)

could have packed the spanish embeddings, but the decoders outputs would have been packed sequences as well, which we wpild have to pad before passing to the output layer. Avoided this complexity because we can configure the loss to ignore output tokens when tatgets are padding tokens like this:

test:

In [68]:
from utils.early_stopping import EarlyStopping

In [69]:
xentropy = nn.CrossEntropyLoss(ignore_index=0, reduction='sum') # ignores the pad tokens

optimizer = torch.optim.NAdam(nmt_model.parameters(), lr=0.001)
early_stopper = EarlyStopping(patience=50, checkpoint_path='nmt.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',patience=2,factor=0.9)

In [70]:
vocab_size

10000

In [71]:
n_epochs = 10

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    nmt_model.train()
    # iterate through the training data
    for inputs,labels in nmt_train_loader:
        inputs,labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = nmt_model(inputs)
        loss =  xentropy(out, labels)
        loss.backward()
        optimizer.step()
        train_loss[epoch] += loss.item()
    train_loss[epoch] /= len(nmt_train_loader.dataset)

    # eval on the val set
    nmt_model.eval()
    with torch.no_grad():
        for inputs, labels in nmt_valid_loader:
            inputs,labels = inputs.to(device),labels.to(device)
            out = nmt_model(inputs)
            loss = xentropy(out, labels)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(nmt_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], nmt_model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break




Epoch: 1| Train loss: 30.3253| Val loss: 23.3344
Metric improved to 23.3344. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 19.6648| Val loss: 21.5679
Metric improved to 21.5679. Checkpoint saved at epoch 1
Epoch: 3| Train loss: 16.5452| Val loss: 21.3851
Metric improved to 21.3851. Checkpoint saved at epoch 2
Epoch: 4| Train loss: 14.9580| Val loss: 21.7580
No improvement for 1 epoch(s)


KeyboardInterrupt: 

load the traiend model..

In [72]:
state_dict = torch.load("nmt.pt")
state_dict

{'epoch': 2,
 'model_state_dict': OrderedDict([('embed.weight',
               tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
                       [-1.3638,  0.1930, -0.6103,  ...,  1.1085,  0.5544,  1.5818],
                       [-1.0313,  1.0298, -1.3911,  ...,  0.3001,  0.2468,  1.2138],
                       ...,
                       [ 1.1340,  0.7136, -0.6686,  ..., -0.0732, -0.3804,  1.0276],
                       [ 0.8922, -0.0352, -0.7937,  ...,  0.1723,  0.3732, -0.0375],
                       [-2.3785,  0.6653, -1.1006,  ..., -2.1197,  0.4477,  2.0945]],
                      device='mps:0')),
              ('encoder.weight_ih_l0',
               tensor([[ 0.1565,  0.2623, -0.1956,  ...,  0.0505,  0.1319,  0.1005],
                       [ 0.1399, -0.1638,  0.0489,  ..., -0.1445, -0.0620,  0.2575],
                       [ 0.0123,  0.1083, -0.1169,  ..., -0.2299, -0.0534, -0.0306],
                       ...,
                       [-0.0330, -0

In [73]:
nmt_model.load_state_dict(state_dict['model_state_dict'])
nmt_model.eval()

NmtModel(
  (embed): Embedding(10000, 512, padding_idx=0)
  (encoder): GRU(512, 512, num_layers=2, batch_first=True)
  (decoder): GRU(512, 512, num_layers=2, batch_first=True)
  (output): Linear(in_features=512, out_features=10000, bias=True)
)

Custom translate function:

In [74]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [75]:
nmt_model.eval()
translate(nmt_model, "I like soccer.")

' me gusta el fútbol . </s>'

In [82]:
translate(nmt_model, "He is not gay")

' no es gay . </s>'

should have used the whitespace pretokenizer

Attention ...
- Adding Luong's attention to the encoder-decoder model
- Luong's attention (multiplicative attention) proposes computing the dot product between the encoder outputs and decoder's current state values instead of using a dense layer for the resultant energy values as in the original concatenative attention technique
- Writing custom attention

In [93]:
a = torch.rand(32, 18, 512) # hypothetical query - hidden states of decoder - B x seq_len x output_dim
b = torch.rand(32, 20, 512) # hypothetical key and value - encoder outputs - shape: B x seq_len x output_dim

In [94]:
(a@b.transpose(1,2)).shape # what we'll be doing first is crossing the queries with the keys
# this computation produces the weights and a way of looking at is saying that for each of the decoder's hidden states
# we have a corresponding weight mapping to each of the encoder's output
# output of this is batch x decoder hidden state seq length x encoder output seq length -> 
# for each of the decoder's hidden state we have a corresponding weight mapping to each of the encoder's outputs

torch.Size([32, 18, 20])

In [95]:
z = a@b.transpose(1,2) # computing the weight matrix

In [96]:
torch.softmax(z, dim=-1).shape # we need to turn the weights to attention weights that sum up to 1
# we use the torch.softmax function for this -> as explained earlier, for each instance (timestep) in the decoder's hidden state output, we have a corresponding 
# weight score for the encoder output sequence

torch.Size([32, 18, 20])

In [97]:
z1 = torch.softmax(z, dim=-1) # softmax weight scores computed here....

In [92]:
(z1@b).shape
# way to think of this now is that we need to use the weight scores to extract that weighted part of the encoder output
# we are now using the weights to extract the corresponding sum (aggregation) from the encoder's output - in particular using the dimensuons

torch.Size([32, 18, 512])

In [99]:
torch.cat((torch.rand(32, 18, 512), torch.rand(32, 18, 512)), dim=-1).shape

torch.Size([32, 18, 1024])

In [100]:
def attention(query, key, value):  # note that dq==dk and Lk == Lv . d (embedding dimension for k-key and q-query). L (length of longest sequence in k-key and v-value)
    scores = query @ key.transpose(1,2) # [B, Lq, dq] @ [B, dk, Lk] == [B, Lq, Lk]
    weights = torch.softmax(scores, dim=-1) # [B, Lq, Lk]
    return weights @ value # [B, Lq, Lk] @ [B, Lv, dv] = [B, Lq, dv]

# this implementation efficiently runs all these computations for the whole batch at once
# the query argument corresponds to the decoder's hidden states - using current timestep as opposed to previous timestep
# the key argument corresponds to the encoder outputs
# the value also corresponds to the encoder outputs

Now build the nmt model with attention...

In [101]:
class NmtModelWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.decoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.output = nn.Linear(2*hidden_dim, vocab_size)
    
    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = nn.utils.rnn.pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        encoder_outputs_packed, hidden_states = self.encoder(src_packed)
        decoder_outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        encoder_outputs, _ = nn.utils.rnn.pad_packed_sequence(encoder_outputs_packed, batch_first=True)
        attn_output = attention(query=decoder_outputs, key=encoder_outputs, value=encoder_outputs)
        combined_output = torch.cat((attn_output, decoder_outputs), dim=-1)
        return self.output(combined_output).permute(0,2,1)

Training the model...

In [102]:
torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model2 = NmtModelWithAttention(vocab_size).to(device)

In [103]:
xentropy = nn.CrossEntropyLoss(ignore_index=0, reduction='sum') # ignores the pad tokens

optimizer = torch.optim.NAdam(nmt_model2.parameters(), lr=0.001)
early_stopper = EarlyStopping(patience=50, checkpoint_path='nmt2.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',patience=2,factor=0.9)

In [104]:
n_epochs = 10

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    nmt_model2.train()
    # iterate through the training data
    for inputs,labels in nmt_train_loader:
        inputs,labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = nmt_model2(inputs)
        loss =  xentropy(out, labels)
        loss.backward()
        optimizer.step()
        train_loss[epoch] += loss.item()
    train_loss[epoch] /= len(nmt_train_loader.dataset)

    # eval on the val set
    nmt_model2.eval()
    with torch.no_grad():
        for inputs, labels in nmt_valid_loader:
            inputs,labels = inputs.to(device),labels.to(device)
            out = nmt_model2(inputs)
            loss = xentropy(out, labels)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(nmt_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], nmt_model2, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break

Epoch: 1| Train loss: 29.4532| Val loss: 23.6862
Metric improved to 23.6862. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 20.8742| Val loss: 22.6000
Metric improved to 22.6000. Checkpoint saved at epoch 1
Epoch: 3| Train loss: 19.0130| Val loss: 22.6477
No improvement for 1 epoch(s)
Epoch: 4| Train loss: 18.0885| Val loss: 22.9598
No improvement for 2 epoch(s)
Epoch: 5| Train loss: 17.5953| Val loss: 23.3647
No improvement for 3 epoch(s)
Epoch: 6| Train loss: 16.6060| Val loss: 23.0898
No improvement for 4 epoch(s)
Epoch: 7| Train loss: 16.0473| Val loss: 23.2877
No improvement for 5 epoch(s)
Epoch: 8| Train loss: 15.8145| Val loss: 23.6536
No improvement for 6 epoch(s)
Epoch: 9| Train loss: 14.9698| Val loss: 23.3960
No improvement for 7 epoch(s)
Epoch: 10| Train loss: 14.5532| Val loss: 23.5548
No improvement for 8 epoch(s)


In [105]:
nmt_model2.eval()
translate(nmt_model2, "I like soccer.")

' me gusta el fútbol . </s>'

In [ ]:
translate(nmt_model2, "Arsenal are playing on Saturday")

' ar and ar el sábado de los sábados . </s>'

In [114]:
translate(nmt_model, "Arsenal are playing on Saturday")

' los jug adores están jugando al sábado . </s>'